<a href="https://colab.research.google.com/github/hrq040107-bit/group5/blob/main/checks_1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib, json, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd


# 🟢 Fill in your group, then run
GROUP = "5"                              # choose a name for your group, e.g. "otters": letters only, used in file names
MEMBERS = ["____", "Anastasiia", ""]  # first names, in the order you will rotate

def check(condition, hint="Not yet."):
    print("✅" if condition else f"❌ {hint}")

CHECKPOINTS = ["Load and verify", "Structure and types", "Text and categories", "Dates", "Duplicates",
               "Units and scales", "Impossible and implausible values", "Missing values (contested)", "Validate and save"]
for i, cp in enumerate(CHECKPOINTS):
    print(f"{i + 1}. {cp:<36} driver: {MEMBERS[i % len(MEMBERS)]}")

1. Load and verify                      driver: ____
2. Structure and types                  driver: Anastasiia
3. Text and categories                  driver: 
4. Dates                                driver: ____
5. Duplicates                           driver: Anastasiia
6. Units and scales                     driver: 
7. Impossible and implausible values    driver: ____
8. Missing values (contested)           driver: Anastasiia
9. Validate and save                    driver: 


In [2]:
# 🟢 Get the file: from the course repository, or upload it
DATA_URL = "https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/refs/heads/main/Week%204/data/week4_country_panel_raw.csv"
PUBLISHED_SHA256 = "116c3194708b7ac967189ddd29a06affc3e7b1071f4ecee10a17e3551ccb9f28"
PATH = Path("week4_country_panel_raw.csv")

if not PATH.exists():
    try:
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, PATH)
    except Exception as err:
        print("Download failed:", type(err).__name__, "- upload the file with the Files panel, then run this cell again.")

print(PATH.read_text(encoding="utf-8").splitlines()[:4])

['# Source: ST445 teaching panel (synthetic, based loosely on published magnitudes)', '# Units: population in millions; GDP per person in current US dollars; health spending as % of GDP', '# Published: 125 country-years (25 countries, 2018-2022), plus 2 revised records', 'country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date']


In [3]:
# 🟢 Verify: checksum, row count, schema
digest = hashlib.sha256(PATH.read_bytes()).hexdigest()
check(digest == PUBLISHED_SHA256, "The file is not the published one: download it again.") # Verify the checksum

raw = pd.read_csv(PATH, comment="#", dtype=str, keep_default_na=False)
EXPECTED_COLUMNS = ["country", "iso3", "year", "population_m", "gdp_pc_usd", "health_spend_pct",
                    "life_expectancy", "income_group", "survey_date"]
check(list(raw.columns) == EXPECTED_COLUMNS, f"Unexpected columns: {list(raw.columns)}") # Verify the number of columns

PUBLISHED_ROWS = 125 + 2 # As mentioned in the metadata in previous cell
print(f"Rows read: {len(raw)} | published: {PUBLISHED_ROWS} | difference: {len(raw) - PUBLISHED_ROWS}")

✅
✅
Rows read: 131 | published: 127 | difference: 4


In [4]:
# 🟢 Look before changing anything: what is in each column that should be numeric?
for col in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]:
    as_number = pd.to_numeric(raw[col], errors="coerce")
    not_numbers = raw.loc[as_number.isna(), col].value_counts()
    print(f"\n{col}: {len(not_numbers)} kinds of non-number")
    print(not_numbers.head(6).to_dict())
    print("range of the values that are numbers:", as_number.min(), "to", as_number.max())

raw.tail(3)


year: 1 kinds of non-number
{'': 1}
range of the values that are numbers: 2018.0 to 2022.0

population_m: 1 kinds of non-number
{'': 1}
range of the values that are numbers: -999.0 to 85400.0

gdp_pc_usd: 23 kinds of non-number
{'3,990': 2, '10,840': 1, '64,920': 1, '2,300': 1, '9,420': 1, '1,630': 1}
range of the values that are numbers: -10220.0 to 72360.0

health_spend_pct: 3 kinds of non-number
{'': 15, 'n/a': 11, '..': 5}
range of the values that are numbers: 2.3 to 17.8

life_expectancy: 2 kinds of non-number
{'n/a': 1, '': 1}
range of the values that are numbers: -1.0 to 824.0


,country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date
128,Bangladesh,BGD,2022,167.5,1800,2.4,72.1,Lower middle income,2023-03-25
129,Pakistan,PAK,2022,223.5,"1,690",3.4,65.7,Lower middle income,2023-03-24
130,Total,,,,,,,,


In [5]:
df = raw.copy()

# (a) Remove rows that are not observations (checkpoint 1 showed one at the end of the file)
df = df[df["country"] != "Total"]

# (b) Strip leading and trailing spaces from every text value
df = df.apply(lambda s: s.str.strip())

# (c) Declare every missing-value marker checkpoint 1 found, and replace them with a real missing value
MARKERS = ["", "-999", "n/a", ".."]          # hint: three more; one of them is a number
df = df.replace({m: pd.NA for m in MARKERS})

# (d) Convert the numeric columns. GDP is written with thousands separators: remove them first
missing_before = df.isna().sum().sum()
df["year"] = df["year"].astype("Int16")          # a small nullable integer, e.g. "Int16"
df["gdp_pc_usd"] = pd.to_numeric(df["gdp_pc_usd"].str.replace(",", "", regex=False), errors="coerce")
for col in ["population_m", "health_spend_pct", "life_expectancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
created = df.isna().sum().sum() - missing_before
print("Rows:", len(df), "| missing values created by conversion:", created)
df.dtypes

Rows: 130 | missing values created by conversion: 0


,0
country,object
iso3,object
year,Int16
population_m,float64
gdp_pc_usd,float64
health_spend_pct,float64
life_expectancy,float64
income_group,object
survey_date,object


In [6]:
# Self-check for checkpoint 2
check(len(df) == 130, "130 rows should remain: only the 'Total' row is removed here.")
check(created == 0, "Conversion created missing values: a marker or a format is still undeclared.")
check(all(pd.api.types.is_numeric_dtype(df[c]) for c in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]),
      "Every numeric column should now be numeric.")

✅
✅
✅


In [7]:
# 🟢 Look first: every spelling of each country, and every income group label
spellings = df.groupby(df["iso3"].str.upper())["country"].unique()
print(spellings[spellings.map(len) > 1])
print()
print(df["income_group"].value_counts())

iso3
CIV    [CÃ´te d'Ivoire, Côte d'Ivoire, Côte d’Ivoire]
GBR              [United Kingdom, UK, united kingdom]
KOR                        [Korea, Rep., South Korea]
TUR                       [Türkiye, Turkey, Türkiye]
Name: country, dtype: object

income_group
High income            41
Lower middle income    30
Upper middle income    23
Low income             12
low income              6
Lower middle            4
Low                     3
lower-middle income     3
High                    2
high income             2
upper middle income     2
High Income             1
Upper-middle            1
Name: count, dtype: int64


In [13]:
# (a) Standardise the key first: upper-case ISO3 codes (spaces were stripped in checkpoint 2)
df["iso3"] = df["iso3"].str.strip().str.upper()

# (b) One canonical name per country, from the standard list (World Bank style names)
ISO3_TO_NAME = {"GBR": "United Kingdom", "FRA": "France", "DEU": "Germany", "ITA": "Italy", "ESP": "Spain",
                "USA": "United States", "JPN": "Japan", "KOR": "Korea, Rep.", "QAT": "Qatar", "CHN": "China",
                "BRA": "Brazil", "MEX": "Mexico", "TUR": "Türkiye", "ZAF": "South Africa", "IDN": "Indonesia",
                "IND": "India", "NGA": "Nigeria", "KEN": "Kenya", "CIV": "Côte d'Ivoire", "PAK": "Pakistan",
                "BGD": "Bangladesh", "ETH": "Ethiopia", "UGA": "Uganda", "MOZ": "Mozambique", "NER": "Niger"}
df["country"] = df["iso3"].map(ISO3_TO_NAME)
check(df["country"].notna().all(), "Some ISO3 codes did not map: print df.loc[df['country'].isna(), 'iso3'].")

# (c) Harmonise income groups: lower-case, hyphens to spaces, add "income" where it is missing,
#     then capitalise and store as an ordered category
INCOME = ["Low income", "Lower middle income", "Upper middle income", "High income"]
inc = df["income_group"].str.lower().str.replace("-", " ", regex=False).str.strip()
inc = inc.where(inc.str.endswith("income"), inc + " income")
df["income_group"] = inc.str.capitalize().astype(pd.CategoricalDtype(INCOME, ordered=True))
check(df["income_group"].notna().all(), "A label did not match one of the four categories: print the unmatched ones.")
df["income_group"].value_counts(sort=False)

✅
✅


,count
income_group,
Low income,21
Lower middle income,37
Upper middle income,26
High income,46


In [14]:
# Self-check for checkpoint 3
check(df.groupby("iso3")["country"].nunique().max() == 1, "Each ISO3 code should have exactly one name.")
check(df["country"].map(lambda x: "Ã" not in x and unicodedata.is_normalized("NFC", x)).all(), "Names should be NFC, with no mojibake.")
check(df["income_group"].nunique() == 4, "There should be exactly four income groups.")

✅
✅
✅
